In [1]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scanpy as sc
import seaborn as sns
import shutil
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from helpers import *

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

In [2]:
# File paths
comparison_path = f"../output/MERSCOPE_WT_AD_comparison/"
output_path = f"../output/MERSCOPE_WT_AD_comparison/" + "neuropil_subdomains/"
shutil.rmtree(output_path, ignore_errors=True)
os.makedirs(output_path, exist_ok=True)

In [3]:
# ==================== Read data ==================== #

# Spots
spots = sc.read_h5ad(comparison_path + "neuropil_subdomains_spots.h5ad")
spots = spots[spots.obs["brain_area"].isin(["Isocortex"])].copy()

# Cells
adata = sc.read_h5ad(comparison_path + "neuropil_subdomains_adata.h5ad")
adata = adata[adata.obs["brain_area"].isin(["Isocortex"])].copy()

# Granules
granule_adata = sc.read_h5ad(comparison_path + "granule_adata_tsne.h5ad")

granule_adata.obs["global_x"] = granule_adata.obs["global_x_adjusted"].copy()
granule_adata.obs["global_y"] = granule_adata.obs["global_y_adjusted"].copy()
mask = granule_adata.obs["batch"] == "MERSCOPE_WT_1"
granule_adata.obs.loc[mask, "global_x"] = (6250 - granule_adata.obs.loc[mask, "global_x"])

granule_subtype_df = pd.read_parquet(comparison_path + "granule_subtype_labels_granule_adata_tsne.parquet")
cols_keep = ["sample", "granule_id", "granule_subtype_kmeans", "granule_subtype_manual", "granule_subtype_manual_simple"]
granule_subtype_df = granule_subtype_df[cols_keep].drop_duplicates(["sample", "granule_id"])

granule_adata.obs = granule_adata.obs.reset_index(names="obs_name")
granule_adata.obs = granule_adata.obs.merge(granule_subtype_df,
                                            left_on=["batch", "granule_id"],
                                            right_on=["sample", "granule_id"],
                                            how="left",
                                            validate="one_to_one").set_index("obs_name")

if "sample" in granule_adata.obs.columns:
    granule_adata.obs = granule_adata.obs.drop(columns=["sample"])

granule_adata.obs["granule_subtype_kmeans"] = granule_adata.obs["granule_subtype_kmeans"].astype("category")
granule_adata.obs["granule_subtype_manual"] = granule_adata.obs["granule_subtype_manual"].astype("category")
granule_adata.obs["granule_subtype"] = granule_adata.obs["granule_subtype_manual_simple"].astype("category")

# granule_adata = granule_adata[granule_adata.obs["brain_area"].isin(["Isocortex"])].copy()

# # Plot check
# sc.pl.scatter(spots, x="global_x", y="global_y", color="batch")
# plt.show()

# sc.pl.scatter(adata, x="global_x", y="global_y", color="batch")
# plt.show()

# sc.pl.scatter(granule_adata, x="global_x", y="global_y", color="batch")
# plt.show()

In [5]:
# # Compute grid-level embeddings
# embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding(
#     spots=spots,
#     granule_adata=granule_adata,
#     adata=adata,
#     spot_width=25,
#     spot_height=25,
#     granule_subtype_key="granule_subtype_kmeans",
#     subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
#     granule_count_layer="counts",
#     include_soma_features=True,
#     smoothing=True,
#     smoothing_radius=np.sqrt(2) * 25 + 1,
#     smoothing_mode="gaussian"
# )

# # embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding_granule_subtype_kernel_grid(
# #         spots=spots,
# #         granule_adata=granule_adata,
# #         adata=adata,
# #         spot_width=25,
# #         spot_height=25,
# #         granule_subtype_key="granule_subtype_kmeans",
# #         subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
# #         granule_count_layer="counts",
# #         include_soma_features=True,
# #         kernel="gaussian",
# #         sigma=12.5,
# #         support_radius=37.5,
# #         normalize_subtype_embedding=True,
# #         normalize_gene_counts=False,
# #     )

# # embeddings, embeddings_features, aux_features, spot_granule_expression = \
# #     spot_embedding_spatial_weight(
# #         spots=spots,
# #         granule_adata=granule_adata,
# #         adata=adata,
# #         spot_width=25,
# #         spot_height=25,
# #         granule_subtype_key="granule_subtype_kmeans",
# #         subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
# #         granule_count_layer="counts",
# #         include_soma_features=True,
# #         # neighbor_mode="grid_window",   # or "radius"
# #         # sigma=5.0,
# #         # kernel="exponential",          # to mimic your old neuron code more closely
# #         neighbor_mode="radius",
# #         radius=10.0,
# #         sigma=5.0,
# #         # kernel="exponential"
# #         include_padding_category=False,
# #         normalize_subtype_embedding=True,
# #         normalize_gene_counts=False,
# #     )

# for aux_key, aux_val in aux_features.items():
#     print(aux_key)
#     spots.obs[aux_key] = aux_val

In [6]:
n_clusters = 4
spots0 = spots.copy()

for embedding_method in ["hard", "soft"]:
    
    spots = spots0.copy()
    
    if embedding_method == "hard":
        
        embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata,
            spot_width=25,
            spot_height=25,
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            include_soma_features=True,
            smoothing=True,
            smoothing_radius=np.sqrt(2) * 25 + 1,
            smoothing_mode="gaussian"
        )
    
    elif embedding_method == "soft":
        
        embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding_granule_subtype_kernel_grid(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata,
            spot_width=25,
            spot_height=25,
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            include_soma_features=True,
            kernel="exponential",
            sigma=12.5,
            support_radius=25,
            normalize_subtype_embedding=False,
            normalize_gene_counts=False,
        )
    
    for aux_key, aux_val in aux_features.items():
        spots.obs[aux_key] = aux_val
    
    mask = (spots.obs["granule_count"] > 0)
    spots = spots[mask].copy()
    embeddings = embeddings[mask].copy()
    spot_granule_expression = spot_granule_expression[mask].copy()

    for mode in ["unnormalized", "normalized"]:
        
        print(f"Embedding method: {embedding_method}, mode: {mode}")
        
        if mode == "normalized":
            row_sums = embeddings.sum(axis=1, keepdims=True)
            X = np.divide(embeddings, row_sums, out=np.zeros_like(embeddings, dtype=float), where=row_sums > 0)
        else:
            X = embeddings
        
        print(X.sum(axis = 1))
        
        # LDA clustering
        lda = LatentDirichletAllocation(n_components = n_clusters, random_state = 42)
        lda_labels = lda.fit_transform(X)
        spots.obs["subdomain_lda"] = [f"Subdomain {l + 1}" for l in np.argmax(lda_labels, axis = 1)]
        
        # GMM clustering
        gmm = GaussianMixture(n_components = n_clusters, random_state = 42)
        gmm_labels = gmm.fit_predict(X)
        spots.obs["subdomain_gmm"] = [f"Subdomain {l + 1}" for l in gmm_labels]
        
        # K-Means clustering
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
        kmeans_labels = kmeans.fit_predict(X)
        spots.obs["subdomain_kmeans"] = [f"Subdomain {l + 1}" for l in kmeans_labels]
        
        # Minibatch K-Means clustering
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=42, n_init=20)
        kmeans_labels = kmeans.fit_predict(X)
        spots.obs["subdomain_minibatch"] = [f"Subdomain {l + 1}" for l in kmeans_labels]
        
        for cluster_col in ["subdomain_lda", "subdomain_gmm", "subdomain_kmeans", "subdomain_minibatch"]:
            
            # Scatter plot
            sc.set_figure_params(scanpy = True, figsize = (10, 8))
            ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=cluster_col, size=15, title=cluster_col, show = False)
            ax.grid(False)
            plt.savefig(output_path + f"{embedding_method}_{mode}_{cluster_col.split('_')[1]}.jpeg", dpi = 500, bbox_inches = "tight")
            plt.close()
            
            for cluster in spots.obs[cluster_col].unique():
                sc.set_figure_params(scanpy = True, figsize = (10, 8))
                ax = sc.pl.scatter(spots[spots.obs[cluster_col] == cluster], x="global_x", y="global_y", color=cluster_col, size=15, title=cluster_col, show = False)
                ax.grid(False)
                plt.savefig(output_path + f"{embedding_method}_{mode}_{cluster_col.split('_')[1]}_{cluster}.jpeg", dpi = 500, bbox_inches = "tight")
                plt.close()
                
            # Embedding heatmap
            subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
            desired_order = [f"Subdomain {i + 1}" for i in range(n_clusters)]

            embedding_df = pd.DataFrame(X, columns = subtype_labels)
            embedding_df[cluster_col] = spots.obs[cluster_col].values
            cluster_means = embedding_df.groupby(cluster_col).mean()
            cluster_means = cluster_means.loc[desired_order, subtype_labels]
            
            plt.figure(figsize = (5, 6))
            sns.heatmap(cluster_means.T, cmap = "Reds", annot = False, linewidths = 0.5, linecolor = "lightgray", xticklabels = True, yticklabels = True)
            plt.grid(False)
            plt.title(" ")
            plt.xlabel("Neuropil Subdomain")
            plt.ylabel("Granule Subtype")
            plt.savefig(output_path + f"{embedding_method}_{mode}_heatmap_{cluster_col.split('_')[1]}.jpeg", dpi = 500, bbox_inches = "tight")
            plt.close()
            
            if mode == "normalized":
            
                global_means = embedding_df[subtype_labels].mean(axis=0)
                fold_mat = cluster_means.divide(global_means, axis=1).T
                eps = 1e-8
                log2fc_mat = np.log2((cluster_means + eps).divide(global_means + eps, axis=1)).T
                
                vmax = np.nanmax(np.abs(log2fc_mat.values))
                vmin = -vmax
                plt.figure(figsize=(5, 6))
                sns.heatmap(log2fc_mat, cmap="bwr", center=0, vmin=vmin, vmax=vmax, linewidths=0.5, linecolor="lightgray")
                plt.grid(False)
                plt.title(" ")
                plt.xlabel("Neuropil Subdomain")
                plt.ylabel("Granule Subtype")
                plt.savefig(output_path + f"{embedding_method}_{mode}_heatmap_log2fc_{cluster_col.split('_')[1]}.jpeg", dpi=500, bbox_inches="tight")
                plt.close()

Embedding method: hard, mode: unnormalized
[2.28336834 1.27547679 3.93159186 ... 2.32552818 3.80065204 3.88174978]
Embedding method: hard, mode: normalized
[1. 1. 1. ... 1. 1. 1.]
Embedding method: soft, mode: unnormalized
[0.41878822 0.13543337 4.22875517 ... 1.86540364 4.52681789 3.99554883]
Embedding method: soft, mode: normalized
[1. 1. 1. ... 1. 1. 1.]


In [ ]:
cluster_col.split("_")[1]

In [ ]:
comp_df = pd.DataFrame(embeddings, columns=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())])
comp_df["batch"] = spots.obs["batch"].values
comp_df.groupby("batch").mean()

In [ ]:
comp_df = pd.DataFrame(embeddings, columns=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())])
comp_df["batch"] = spots.obs["batch"].values
comp_df.groupby("batch").mean()

In [ ]:
n_clusters = 5
# LDA clustering on embeddings
lda = LatentDirichletAllocation(n_components = n_clusters, random_state = 42)
lda_labels = lda.fit_transform(embeddings)

In [ ]:
spots.obs["subdomain_lda"] = [f"Substate {l + 1}" for l in np.argmax(lda_labels, axis = 1)]

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="subdomain_lda", size=15, title="subdomain_lda", show = False)
ax.grid(False)
plt.show()

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots[spots.obs["subdomain_lda"] == "Substate 1"], x="global_x", y="global_y", color="subdomain_lda", size=15, title="subdomain_lda", show = False)
ax.grid(False)
plt.show()

In [ ]:
del spots.uns["subdomain_lda_colors"]

In [ ]:
# Embedding heatmaps
cluster_col = "subdomain_lda"
subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
desired_order = [f"Substate {i + 1}" for i in range(n_clusters)]

embedding_df = pd.DataFrame(embeddings, columns = subtype_labels)
embedding_df[cluster_col] = spots.obs[cluster_col].values
cluster_means = embedding_df.groupby(cluster_col).mean()
subtype_order = [f"{i}" for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
cluster_means = cluster_means.loc[desired_order, subtype_order]

plt.figure(figsize = (5, 6))
sns.heatmap(cluster_means.T, cmap = "Reds", annot = False, linewidths = 0.5, linecolor = "lightgray", xticklabels = True, yticklabels = True)
plt.grid(False)
plt.title(" ")
plt.xlabel("Neuron Cluster")
plt.ylabel("Granule Subtype")
plt.show()

In [ ]:
eps = 1e-6

wt_mask = spots.obs["batch"].to_numpy() == "MERSCOPE_WT_1"
ad_mask = spots.obs["batch"].to_numpy() == "MERSCOPE_AD_1"

if wt_mask.sum() == 0:
    raise ValueError(f"No spots found with")
if ad_mask.sum() == 0:
    raise ValueError(f"No spots found with")

# WT mean composition baseline
wt_mean_prop = embeddings[wt_mask].mean(axis=0)

In [ ]:
# =========================================================
# 5. Summarize WT-defined GMM domains by composition
# =========================================================
prop_df = pd.DataFrame(embeddings, columns=subtype_labels)
prop_df["cluster"] = np.argmax(lda_labels, axis = 1)
prop_df["condition"] = spots.obs["batch"].to_numpy()

# mean subtype proportions per WT-defined domain
cluster_profile = prop_df.groupby("cluster")[subtype_labels].mean()

# enrichment relative to global mean across all spots
global_mean = prop_df[subtype_labels].mean(axis=0)
cluster_enrichment_global = cluster_profile / global_mean.values[None, :]
cluster_enrichment_global.columns = [f"global_enrich_{x}" for x in subtype_labels]

# enrichment relative to WT baseline
cluster_enrichment_wt = cluster_profile / wt_mean_prop[None, :]
cluster_enrichment_wt.columns = [f"wt_enrich_{x}" for x in subtype_labels]

cluster_summary = pd.concat(
    [cluster_profile, cluster_enrichment_global, cluster_enrichment_wt],
    axis=1
).sort_index()

print("\nWT-reference GMM domain summary:")
print(cluster_summary.round(4))

# identify the most pre-syn-enriched WT-defined domain
presyn_cluster = cluster_summary["wt_enrich_pre-syn"].idxmax()
print(f"\nMost pre-syn-enriched WT-reference domain: cluster {presyn_cluster}")

spots.obs["is_presyn_enriched_domain"] = (np.argmax(lda_labels, axis = 1) == presyn_cluster).astype(int)

# optional: rank domains by pre-syn enrichment
ranked_domains = cluster_summary.sort_values("wt_enrich_pre-syn", ascending=False)
print("\nDomains ranked by WT-relative pre-syn enrichment:")
print(ranked_domains[["pre-syn", "wt_enrich_pre-syn"]].round(4))

# =========================================================
# Optional quick summaries for checking
# =========================================================

# A. fraction of spots in each WT-reference domain by condition
domain_freq = pd.crosstab(
    spots.obs["batch"],
    spots.obs["subdomain_gmm_wtref"],
    normalize="index"
)
print("\nWT-reference domain frequencies by condition:")
print(domain_freq.round(4))

# B. fraction of spots in the pre-syn-enriched domain by condition
presyn_domain_freq = spots.obs.groupby("batch")["is_presyn_enriched_domain"].mean()
print("\nFraction of spots in pre-syn-enriched WT-reference domain:")
print(presyn_domain_freq.round(4))

# C. mean continuous enrichment scores by condition
enrichment_cols = [f"enrich_{s}" for s in subtype_labels] + ["score_pre_vs_rest"]
enrichment_summary = spots.obs.groupby("batch")[enrichment_cols].mean()
print("\nMean continuous enrichment scores by condition:")
print(enrichment_summary.round(4))

In [ ]:
# GMM clustering on embeddings
gmm = GaussianMixture(n_components = n_clusters, random_state = 42)
gmm_labels = gmm.fit_predict(embeddings)
spots.obs["subdomain_gmm"] = [f"Substate {l + 1}" for l in gmm_labels]

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="subdomain_gmm", size=15, title="subdomain_gmm", show = False)
ax.grid(False)
plt.show()

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots[spots.obs["subdomain_gmm"] == "Substate 2"], x="global_x", y="global_y", color="subdomain_gmm", size=15, title="subdomain_gmm", show = False)
ax.grid(False)
plt.show()

In [ ]:
# Embedding heatmaps
cluster_col = "subdomain_gmm"
subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
desired_order = [f"Substate {i + 1}" for i in range(n_clusters)]

embedding_df = pd.DataFrame(embeddings, columns = subtype_labels)
embedding_df[cluster_col] = spots.obs[cluster_col].values
cluster_means = embedding_df.groupby(cluster_col).mean()
subtype_order = [f"{i}" for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
cluster_means = cluster_means.loc[desired_order, subtype_order]

plt.figure(figsize = (5, 6))
sns.heatmap(cluster_means.T, cmap = "Reds", annot = False, linewidths = 0.5, linecolor = "lightgray", xticklabels = True, yticklabels = True)
plt.grid(False)
plt.title(" ")
plt.xlabel("Neuron Cluster")
plt.ylabel("Granule Subtype")
plt.show()

In [ ]:
embeddings.sum(axis = 1)

In [ ]:
wt_mask = spots.obs["batch"].to_numpy() == "MERSCOPE_WT_1"
ad_mask = spots.obs["batch"].to_numpy() == "MERSCOPE_AD_1"

X_wt = embeddings[wt_mask]
X_ad = embeddings[ad_mask]
X_all = embeddings

In [ ]:
# K-Means clustering on embeddings
n_clusters = 5

kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=0, n_init=20)
kmeans.fit(embeddings)
spots.obs["subdomain_kmeans"] = [f"Substate {l + 1}" for l in kmeans.labels_]

# # WT
# kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=0, n_init=20)
# kmeans.fit(X_wt)
# spots.obs.loc[wt_mask, "subdomain_kmeans"] = kmeans.labels_.astype(str)

# # AD
# kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=0, n_init=20)
# kmeans.fit(X_ad)
# spots.obs.loc[ad_mask, "subdomain_kmeans"] = kmeans.labels_.astype(str)

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="subdomain_kmeans", size=15, title="subdomain_kmeans", show = False)
ax.grid(False)
plt.show()

In [ ]:
ax = sc.pl.scatter(spots[spots.obs["subdomain_kmeans"] == "Substate 1"], x="global_x", y="global_y", color="subdomain_kmeans", size=15, title="subdomain_kmeans", show = False)
ax.grid(False)
plt.show()

In [ ]:
# Embedding heatmaps
cluster_col = "subdomain_kmeans"
subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
desired_order = [f"Substate {i + 1}" for i in range(n_clusters)]

embedding_df = pd.DataFrame(embeddings, columns = subtype_labels)
embedding_df[cluster_col] = spots.obs[cluster_col].values
cluster_means = embedding_df.groupby(cluster_col).mean()
subtype_order = [f"{i}" for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
cluster_means = cluster_means.loc[desired_order, subtype_order]

plt.figure(figsize = (5, 6))
sns.heatmap(cluster_means.T, cmap = "Reds", annot = False, linewidths = 0.5, linecolor = "lightgray", xticklabels = True, yticklabels = True)
plt.grid(False)
plt.title(" ")
plt.xlabel("Neuron Cluster")
plt.ylabel("Granule Subtype")
plt.show()

In [ ]:
# Evaluate ARI against K-Means on (1) granule expression and (2) extrasomatic expression

for data, label in zip([spots.X.copy(), spots.layers["extrasomatic_transcripts"].copy(), spot_granule_expression], ["total_expr", "extrasomatic_expr", "granule_expr"]):
    if not isinstance(data, np.ndarray):
        data = data.toarray()
    sums = data.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1
    data = data / sums * 1e4
    data = np.log1p(data)
    kmeans = MiniBatchKMeans(n_clusters=K, batch_size=5000, random_state=0, n_init=20)
    kmeans.fit(data)
    spots.obs[f"{label}_kmeans"] = kmeans.labels_.astype(str)
    print(f"ARI between {label} and subdomain: {adjusted_rand_score(spots.obs[f'{label}_kmeans'], spots.obs['subdomain_gmm_wtref']):.6f}")
    
    ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=f"{label}_kmeans", size=5.5, title=f"{label}_kmeans", show=False)
    plt.savefig(output_path + f"{label}_kmeans.png", dpi = 500, bbox_inches = "tight")
    plt.close()